# 駐車危険場所検索ツール　メンバー：だいち　ゆうせい　○○


In [1]:
# 読み込むファイルを設定する（Notebookと同じフォルダに保存）

filename = "bike.csv"
encoding = "cp932"

# 千葉県警のcsvはShift-JIS（CP932）。追加インストールは不要

import csv

with open(filename, encoding=encoding, newline="") as file:
    incidents = list(csv.DictReader(file))

# 1行が1件の事件。件数を確認する
len(incidents)

10054

# 自転車盗難が多い場所を見つける

このNotebookでは「自転車にどんな対策をするか」ではなく、「どこで盗難が多く発生しているか」を調べます。

市区町村、町丁目、発生場所の種類ごとに盗難件数を集計し、件数が多い場所をランキングで表示します。

※このデータは盗難の発生記録です。各場所の駐輪台数や利用者数は含まれていないため、表示するのは盗難件数の多さです。厳密な盗難確率や発生率ではありません。

## 1. データの件数を確認する

In [2]:
print("読み込んだ盗難データ：", len(incidents), "件")
print("対象の都道府県：", incidents[0]["都道府県（発生地）"])

読み込んだ盗難データ： 10054 件
対象の都道府県： 千葉県


## 2. 盗難件数が多い市区町村をランキングにする

In [3]:
city_counts = {}

for incident in incidents:
    city = incident["市区町村（発生地）"]
    city_counts[city] = city_counts.get(city, 0) + 1

print("【盗難件数が多い市区町村 TOP10】")

for city, count in sorted(city_counts.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(city, ":", count, "件")

【盗難件数が多い市区町村 TOP10】
船橋市 : 1299 件
市川市 : 871 件
松戸市 : 725 件
千葉市中央区 : 708 件
柏市 : 564 件
習志野市 : 528 件
八千代市 : 455 件
浦安市 : 407 件
千葉市美浜区 : 370 件
千葉市稲毛区 : 358 件


## 3. 盗難件数が多い町丁目をランキングにする
市区町村より細かい地域を確認します。

In [4]:
area_counts = {}

for incident in incidents:
    city = incident["市区町村（発生地）"]
    area = incident["町丁目（発生地）"]
    place_name = city + " " + area
    area_counts[place_name] = area_counts.get(place_name, 0) + 1

print("【盗難件数が多い町丁目 TOP15】")

for area, count in sorted(area_counts.items(), key=lambda x: x[1], reverse=True)[:15]:
    print(area, ":", count, "件")

【盗難件数が多い町丁目 TOP15】
船橋市 本町７丁目 : 106 件
習志野市 津田沼１丁目 : 98 件
千葉市緑区 おゆみ野３丁目 : 95 件
松戸市 松戸 : 83 件
八街市 八街ほ : 77 件
浦安市 北栄１丁目 : 67 件
千葉市美浜区 高洲３丁目 : 66 件
八千代市 緑が丘２丁目 : 61 件
成田市 馬橋 : 57 件
野田市 山崎 : 56 件
千葉市若葉区 都賀３丁目 : 56 件
千葉市中央区 富士見２丁目 : 55 件
千葉市中央区 南町２丁目 : 54 件
木更津市 富士見１丁目 : 49 件
東金市 東金 : 48 件


## 4. 気になる市区町村の危険エリアを調べる

市区町村名を入力すると、その地域の町丁目を盗難件数の多い順に表示します。下の `target_city` の文字を変更して実行してください。例：「柏市」「船橋市」「松戸市」

In [5]:
target_city = "柏市"

target_counts = {}

for incident in incidents:
    if incident["市区町村（発生地）"] == target_city:
        area = incident["町丁目（発生地）"]
        target_counts[area] = target_counts.get(area, 0) + 1

print("【" + target_city + "で盗難件数が多い町丁目 TOP10】")

if len(target_counts) == 0:
    print("該当する市区町村が見つかりません。市区町村名を確認してください。")
else:
    for area, count in sorted(target_counts.items(), key=lambda x: x[1], reverse=True)[:10]:
        print(area, ":", count, "件")

【柏市で盗難件数が多い町丁目 TOP10】
根戸 : 37 件
旭町１丁目 : 31 件
中央町 : 21 件
あけぼの１丁目 : 18 件
若柴 : 18 件
大島田１丁目 : 16 件
新柏１丁目 : 16 件
末広町 : 16 件
泉町 : 15 件
柏５丁目 : 15 件


## 5. どのような場所で盗難が多いか調べる
駐車・駐輪場、住宅、道路上など、発生場所の種類を比べます。

In [6]:
place_counts = {}

for incident in incidents:
    place = incident["発生場所"]
    place_counts[place] = place_counts.get(place, 0) + 1

print("【発生場所の種類別 盗難件数】")

for place, count in sorted(place_counts.items(), key=lambda x: x[1], reverse=True):
    print(place, ":", count, "件")

【発生場所の種類別 盗難件数】
その他 : 3297 件
駐車（輪）場 : 3090 件
その他の住宅（３階建て以下共同住宅等） : 1289 件
４階建て以上共同住宅 : 1063 件
一戸建住宅 : 791 件
道路上 : 524 件


## 6. 選んだ市区町村で、どの種類の場所が多いか調べる

In [7]:
target_place_counts = {}

for incident in incidents:
    if incident["市区町村（発生地）"] == target_city:
        place = incident["発生場所"]
        target_place_counts[place] = target_place_counts.get(place, 0) + 1

print("【" + target_city + "の発生場所別 盗難件数】")

for place, count in sorted(target_place_counts.items(), key=lambda x: x[1], reverse=True):
    print(place, ":", count, "件")

【柏市の発生場所別 盗難件数】
その他 : 203 件
駐車（輪）場 : 139 件
その他の住宅（３階建て以下共同住宅等） : 97 件
一戸建住宅 : 67 件
４階建て以上共同住宅 : 38 件
道路上 : 20 件


## 7. 分析結果の見方

- 市区町村ランキング：どの市や区で盗難の記録が多いか
- 町丁目ランキング：同じ市区町村の中で、どの地域に盗難が集中しているか
- 発生場所ランキング：駐輪場、住宅、道路など、どの種類の場所で多いか

件数が多い場所は、駐輪場所を選ぶときに注意して確認したい地域です。ただし、駐輪台数の違いは考慮していないため、「件数が多い＝1台あたりの危険性が高い」とは限りません。